# E9 · Stage 3：10×10 小棋盘 CNN 课程学习（Colab L4 GPU）

本 notebook 实现 `log.md` 待办清单中的 **P0 实验（E9）**：
通过将 20×20 棋盘缩小为 10×10，大幅提升食物密度与随机遭遇率，验证 **全棋盘 CNN（BoardQNet）** 是否能在纯 RL + 稀疏奖励下跳出局部最优陷阱，学会找食物。

## 实验假设与对比

| 项 | Stage 3 之前（E6 / E7 失败教训） | 本实验 E9（10×10 课程学习） |
| --- | --- | --- |
| 棋盘尺寸 | 20×20（400 格，稀疏度极高） | **10×10**（100 格，食物密度高 4 倍） |
| 探索调度 | epsilon 0.4 → 0.05 / 1500 局 | **epsilon 1.0 → 0.05 / 600 局**（前期充分随机探索） |
| 超时控制 | timeout factor 100（前期 300 步才超时） | **timeout factor 50**（前期 150 步，大幅抑制无效转圈） |
| 硬件设备 | Laptop RTX 5060 (6GB) | **NVIDIA L4 (24GB VRAM)** |
| 批大小 / 回放 | batch 128/256, replay 20k/50k | **batch 256, replay 50k, 每 2 步更新一次** |
| 训练规模 | 2486 局（45 分钟超时，eval 0.00） | **1500 局**（预计 5~8 分钟完成） |
| 核心目标 | 解决“绕圈苟命到超时”的死锁 | **观察 score 是否出现上升拐点，验证 CNN 的找食物能力** |


In [ ]:
#@title 1. 环境准备 & GPU 验证 & 项目解压
import os, sys, subprocess, pathlib

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ["OMP_NUM_THREADS"] = "4"

CONTENT = pathlib.Path("/content")
PROJECT = CONTENT / "snake_rl"
TARBALL = CONTENT / "snake_rl_colab.tar.gz"

if not (PROJECT / "pyproject.toml").exists():
    if not TARBALL.exists():
        print("💡 未在 /content 找到 snake_rl_colab.tar.gz，请点击下方按钮选择本地压缩包上传：")
        try:
            from google.colab import files
            uploaded = files.upload()
        except Exception as e:
            raise FileNotFoundError(f"缺少 {TARBALL}，请在左侧文件栏上传，或通过 files.upload() 上传。错误: {e}")
    print("解压项目代码...")
    subprocess.run(["tar", "xzf", str(TARBALL), "-C", str(CONTENT)], check=True)

os.chdir(PROJECT)
print(f"{'cwd':<16}= {os.getcwd()}")
print(f"{'python':<16}= {sys.version.split()[0]}")
print(f"{'cpu_count':<16}= {os.cpu_count()}")
print(f"{'OMP_NUM_THREADS':<16}= {os.environ.get('OMP_NUM_THREADS')}")

# 安装项目（开发模式）
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("\n== GPU 硬件检查 ==")
cuda_ok = torch.cuda.is_available()
print(f"{'cuda_available':<16}= {cuda_ok}")
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"{'gpu_name':<16}= {gpu_name}")
    print(f"{'gpu_vram':<16}= {vram_gb:.2f} GB")
else:
    print("警告: 未启用 GPU！请在 Colab 菜单: 代码执行程序 -> 更改运行时类型 -> 选择 T4 或 L4 GPU")


In [ ]:
#@title 2. 启动 10×10 全棋盘 CNN 训练（L4 优化配置）
import os, subprocess, sys, time

# 核心训练命令参数（针对 L4 GPU 与 10×10 优化）
cmd = [
    sys.executable, "scripts/train_dqn.py",
    "--obs-type", "board",
    "--board-channels", "5",
    "--width", "10",
    "--height", "10",
    "--timeout-factor", "50",
    "--episodes", "1500",
    "--eval-every", "50",
    "--eval-episodes", "20",
    "--final-eval-episodes", "100",
    "--checkpoint-every", "100",
    "--log-every", "25",
    "--batch-size", "256",
    "--max-memory", "50000",
    "--min-replay-size", "1000",
    "--replay-update-every", "2",
    "--epsilon-start", "1.0",
    "--epsilon-end", "0.05",
    "--epsilon-decay-games", "600",
    "--use-target-network",
    "--target-sync-every", "1000",
    "--double-dqn",
    "--device", "cuda",
    "--output-dir", "runs/stage3_board_10x10_seed0",
]

print("执行命令:\n" + " \\\n  ".join(cmd))
print("\n开始训练...\n")
t0 = time.perf_counter()
subprocess.run(cmd, check=True, env={**os.environ, "OMP_NUM_THREADS": "4"})
train_time = time.perf_counter() - t0
print(f"\n训练总耗时: {train_time:.1f}s ({train_time/60:.2f} min)")


In [ ]:
#@title 3. 独立复核评测（100 局 greedy @ seed=10,000）
import subprocess, sys

print("正在运行 100 局 greedy 评估（seed=10,000）...")
subprocess.run(
    [
        sys.executable, "scripts/evaluate.py",
        "--run-dir", "runs/stage3_board_10x10_seed0",
        "--episodes", "100",
        "--seed", "10000",
        "--device", "cuda",
        "--output", "runs/stage3_board_10x10_seed0/eval_10k.json",
    ],
    check=True,
)


In [ ]:
#@title 4. 评测结果与基线对照（E6/E7 vs E9）
import json, pathlib

run = pathlib.Path("runs/stage3_board_10x10_seed0")
summary = json.loads((run / "summary.json").read_text())
final_20k = json.loads((run / "final_eval.json").read_text())
eval_10k = json.loads((run / "eval_10k.json").read_text())
config = json.loads((run / "config.json").read_text())

def fmt_eval(name, r):
    endings = " ".join(f"{k}={v}" for k, v in sorted(r["endings"].items()))
    return (
        f"{name:<12} mean={r['mean_score']:6.2f}  median={r['median_score']:5.1f}  "
        f"max={r['max_score']:3d}  std={r['std_score']:5.2f}  "
        f"steps={r['mean_steps']:6.1f}  endings[{endings}]"
    )

print("== E9 最终评测 (10×10 Board CNN) ==")
print(fmt_eval("colab-20k", final_20k))
print(fmt_eval("colab-10k", eval_10k))
print()
print("== 历史参考对比 (20×20 棋盘) ==")
print("E6 (500 ep short) : mean= 0.17  endings[timeout=44, wall=56]")
print("E7 (45 min long)  : mean= 0.00  endings[timeout=100] (完全绕圈)")
print()
print("== 训练吞吐与成本 ==")
print(f"{'train_steps':<18}= {summary['train_steps']}")
print(f"{'train_seconds':<18}= {summary['train_seconds']:.1f}s ({summary['train_seconds']/60:.2f} min)")
print(f"{'steps_per_second':<18}= {summary['steps_per_second']:.1f}")
print(f"{'replay_updates':<18}= {summary['replay_updates']}")
print(f"{'target_syncs':<18}= {summary['target_syncs']}")


In [ ]:
#@title 5. 绘制训练曲线与死因分布
import csv, pathlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

run = pathlib.Path("runs/stage3_board_10x10_seed0")

def read_csv(name):
    with (run / name).open(encoding="utf-8") as f:
        return list(csv.DictReader(f))

evals = read_csv("eval.csv")
train = read_csv("train.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Stage 3 (E9): 10x10 Board CNN Training Curves", fontsize=14)

# 1. 评分曲线
ep_eval = [int(r["episode"]) for r in evals]
axes[0, 0].plot(ep_eval, [float(r["mean_score"]) for r in evals], "o-", color="#d62728", label="eval mean (20 eps)")
ep_train = [int(r["episode"]) for r in train]
axes[0, 0].plot(ep_train, [float(r["mean100"]) for r in train], color="#1f77b4", label="train mean100")
axes[0, 0].set_title("Score Progression")
axes[0, 0].set_xlabel("Episode")
axes[0, 0].set_ylabel("Score")
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

# 2. 探索率 (Epsilon)
axes[0, 1].plot(ep_train, [float(r["epsilon"]) for r in train], color="orange", label="Epsilon")
axes[0, 1].set_title("Exploration Rate (Epsilon)")
axes[0, 1].set_xlabel("Episode")
axes[0, 1].set_ylabel("Epsilon")
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

# 3. 损失 (TD Loss)
losses = [float(r["long_loss"]) if r["long_loss"] else 0.0 for r in train]
axes[1, 0].plot(ep_train, losses, color="#9467bd", alpha=0.7, label="Replay Loss (MSE)")
axes[1, 0].set_title("TD Loss")
axes[1, 0].set_xlabel("Episode")
axes[1, 0].set_ylabel("Loss")
axes[1, 0].set_yscale("symlog", linthresh=0.1)
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

# 4. 局终死因变化 (每 50 局窗口)
window = 50
ep_blocks = []
wall_ratios = []
self_ratios = []
timeout_ratios = []
for i in range(0, len(train), window):
    chunk = train[i : i + window]
    if not chunk: continue
    ep_blocks.append(int(chunk[-1]["episode"]))
    n = len(chunk)
    wall_ratios.append(sum(int(r["wall"]) for r in chunk) / n)
    self_ratios.append(sum(int(r["self"]) for r in chunk) / n)
    timeout_ratios.append(sum(int(r["timeout"]) for r in chunk) / n)

axes[1, 1].plot(ep_blocks, wall_ratios, label="Wall Collision %", color="red")
axes[1, 1].plot(ep_blocks, self_ratios, label="Self Collision %", color="blue")
axes[1, 1].plot(ep_blocks, timeout_ratios, label="Timeout (Looping) %", color="gray", linestyle="--")
axes[1, 1].set_title("Ending Reason Distribution (50-ep window)")
axes[1, 1].set_xlabel("Episode")
axes[1, 1].set_ylabel("Ratio")
axes[1, 1].set_ylim(-0.05, 1.05)
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

fig.tight_layout()
out_fig = run / "e9_curves.png"
fig.savefig(out_fig, dpi=140, bbox_inches="tight")
plt.close(fig)

display(Image(filename=str(out_fig)))
print("Saved curves to:", out_fig)


## 实验结果解读标准

1. **核心成功指标**：
   - `mean_score` 是否明显超越随机基线（例如 > 3~5 分，10×10 棋盘理论上限为 97）。
   - `timeout` 占比是否明显下降（在 20×20 长跑中是 100% timeout）。如果转变为吃了几颗食物后因身体过长而 `self-collision` 或走投无路撞墙，说明**智能体真正打破了绕圈苟命，开始找食物**！
2. **如果成功**：
   - 证明全棋盘 CNN 结构 + DQN 算法链路是完全正确的，之前的阻碍确实是探索与状态空间稀疏性。
   - 下一步即可开启 P1：扩回 20×20（或改为全卷积 + Global Average Pooling 迁移到任意尺寸）。
3. **如果仍不理想**：
   - 可进一步调整 `timeout_reward`（如设为更强负反馈）或增加奖励塑形（Potential-based shaping）。
